# Day 2 — Tool/Memory Validation, Multi-Hop Reasoning & Failure-Path Testing

**Module 6 · Agentic RAG Testing**

---

## What we'll cover today

| # | Topic | Why it matters |
|---|---|---|
| 1 | Why Module 5's RAGAS suite isn't enough here | Faithfulness/relevancy/precision/recall each have a specific blind spot once retrieval can loop |
| 2 | Memory validation across hops | Same boundary-value problem as Module 5's chunk-boundary bug, relocated to conversation memory |
| 3 | Hard negative: right facts, wrong combination | A failure mode per-fact faithfulness checks structurally cannot catch |
| 4 | Failure-path testing | What should happen when a hop comes up empty — the exact Klarna gap, testable |
| 5 | Extending the coverage matrix | New columns: `reasoning_chain_break`, `premature_stop`, `ungraceful_failure` |

**Estimated time:** 60 minutes
**Run order:** top to bottom. Several cells call the real agent from Day 1 -- make sure `.env` is configured in this `examples/` folder first (see Day 1's setup note).

---

> **Where we are in the course**
> Day 1 built a working, traced 2-hop loop and named 4 new failure modes that only exist in multi-step retrieval.
> Module 5 Day 2 taught boundary value analysis on chunk size; Module 5 Day 3 taught hard negatives for faithfulness.
> Today both come back, pointed at the planner's memory and its reasoning across hops instead of a single retrieval.

---
## Why Day 1's agent needs more than Module 5's test suite

Point Module 5's 4 RAGAS metrics at `agentic_rag()`'s final `response` and its accumulated `retrieved_contexts`, and three of them will happily score a *wrong* answer as fine:

- **`faithfulness`** asks "is every claim in the response backed by *some* retrieved chunk?" — it does not ask "was that the *right* chunk to base the answer on." Today's `reasoning_chain_break` hard negative cites WidgetPro 2000's real, retrieved, individually-true $50 fee — faithfulness has no way to know the question needed WidgetPro 3000's fee instead.
- **`answer_relevancy`** asks "does the response address the question asked?" — a confident, on-topic, wrong answer scores exactly the same as a confident, on-topic, right one.
- **`context_precision` / `context_recall`** ask about the *retrieved* chunks, not about what the generator did with them — they can see a retrieval error, but not a reasoning error layered on top of a clean retrieval.

None of this is a flaw in RAGAS. These metrics were built for a system where there's only ever one fact set to reason over — "did it combine multiple facts correctly" isn't a meaningful question when there's nothing to combine. It only becomes askable once retrieval can happen more than once, which is exactly what Day 1 added. Today builds the three checks that fill that gap:

1. **Memory validation** — did a fact from an early hop survive to the final answer, or get dropped along the way?
2. **Reasoning-combination checks** — were the *right* facts picked and combined, not just retrieved?
3. **Graceful-failure checks** — did the agent admit a gap instead of guessing, when a hop genuinely came up empty?

None of these three have a Module 5 equivalent — not because Module 5's authors missed them, but because none of them could occur in a system that only ever retrieves once.


---
## Memory validation: did the early fact survive?

A multi-hop chain passes facts from early hops forward into the final generation step. Module 4 Day 4 taught you to boundary-test a context window; this is the same boundary, relocated: **does the fact from hop 1 still make it into the final answer, or does it get dropped or corrupted by the time hop 3 generates a response?**

> **Plain English:** this is the chunk-boundary bug from Module 5 Day 2, except the "chunk" is the running memory of a multi-hop conversation instead of a document. The boundary is wherever your agent truncates history — and it can split a needed fact out exactly the same way a 500-token chunk boundary could.

In [1]:
import json
from pydantic import BaseModel
from agent import agentic_rag, judge_client, judge_model

# Load all golden cases once — keyed by id, reused across all 3 checks below.
with open("golden_dataset.json") as f:
    _golden = {c["id"]: c for c in json.load(f)}


---
## Hard negative — right facts, wrong combination (`reasoning_chain_break`)

This is the failure mode Module 5's single-hop metrics structurally cannot catch, because both retrieved facts are individually faithful to their source — the bug is in how they were *combined*.

---
## Failure-path testing: the hop that comes up empty

What should the agent do when a hop finds nothing relevant?

- **Graceful** — admits the gap: *"I found that WidgetPro 3000 replaced WidgetPro 2000, but I don't have its cancellation fee on file."*
- **Ungraceful** — confidently invents a number to fill the gap.

This hard negative is aimed squarely at the failure pattern behind Klarna's "complex cases dropped in quality" from Day 1 — an agent that can't find the next fact should say so, not fabricate one to keep the chain moving.

---
## Extending the coverage matrix

Module 5 Day 2 added retrieval columns to Module 4 Day 4's matrix. Today adds the agentic ones -- and they map directly onto this notebook's opening argument: `reasoning_chain_break` is the check for the failure faithfulness can't see, `ungraceful_failure` is the check for the failure no RAGAS metric asks about at all.


In [5]:
import json
from collections import Counter

with open("golden_dataset.json") as f:
    golden_cases = json.load(f)

categories    = sorted({c["category"] for c in golden_cases})
failure_modes = sorted({c["failure_mode"] for c in golden_cases})
counts        = Counter((c["category"], c["failure_mode"]) for c in golden_cases)

header = " " * 16 + "".join(f"{fm:<24}" for fm in failure_modes)
print(header)
for cat in categories:
    row = f"{cat:<16}" + "".join(f"{counts[(cat, fm)]:<24}" for fm in failure_modes)
    print(row)

zero_cells = [(cat, fm) for cat in categories for fm in failure_modes if counts[(cat, fm)] == 0]
print()
if zero_cells:
    print("Still-empty cells (not necessarily a problem -- just visible now):")
    for cat, fm in zero_cells:
        print(f"- {cat} x {fm}")
else:
    print("No empty cells for these categories x failure modes -- Day 1's named premature_stop gap")
    print("is filled (see multi-hop-turbomax-01 in golden_dataset.json). single_hop_qa correctly")
    print("has no agentic-failure rows -- those columns only apply once a question needs multiple hops.")


                hallucination           premature_stop          reasoning_chain_break   ungraceful_failure      
multi_hop_qa    2                       2                       3                       2                       
single_hop_qa   2                       0                       0                       0                       

Still-empty cells (not necessarily a problem -- just visible now):
- single_hop_qa x premature_stop
- single_hop_qa x reasoning_chain_break
- single_hop_qa x ungraceful_failure


---
## Try It Yourself

1. Write a third memory-validation case where the final answer reflects hop 1's fact but **drops hop 2's entirely**. Does `facts_present_in_answer()` catch it the same way it caught the fully-dropped case above?
2. Add a `query_drift` row to `golden_dataset.json` -- a case where the reformulated query in hop 2 wanders away from the original question's intent (`query_drift` is named in Day 1's failure-mode table but has no row yet). What would its `retrieved_contexts` need to look like for a hard negative to actually demonstrate the drift, rather than just a wrong answer?
3. Write your own "right facts, wrong combination" hard negative in a domain other than product fees (e.g. dates, locations, prices) and add it to `golden_dataset.json` with `eval_type: "reasoning"`, `must_include`, and `must_not_include` fields (see `multi-hop-widgetpro-01` for the pattern). What made it easy or hard to construct compared to the WidgetPro example?

Exercise file: [`exercises/02_tool_memory_reasoning_exercise.md`](../exercises/02_tool_memory_reasoning_exercise.md)


---
## Summary

### What we built today
- A memory-validation check that catches facts dropped between hops — run against the REAL agent's answer, not a scripted stand-in — the conversational version of Module 5's chunk-boundary bug
- A `reasoning_chain_break` hard negative that a per-fact faithfulness check would have missed entirely
- A graceful-vs-ungraceful failure check aimed at the exact gap behind the Klarna incident
- A coverage matrix extended with `reasoning_chain_break`, `premature_stop`, and `ungraceful_failure`

### The thread through this whole module
Every check built across these two days reused a Module 4/5 technique and pointed it one level up: equivalence partitioning -> hop count, boundary value analysis -> conversation memory, hard negatives -> reasoning combination and graceful failure, coverage matrix -> agentic failure modes. Same mindset, new surface — same habit this course keeps building.

**Next:** Module 7 — AI Agents Testing with DeepEval, where these hand-rolled checks become formal, reusable metrics (task completion, tool correctness, argument correctness) and the agent gains real tool-calling, not just retrieval.

---